# LeetCode 3948 — Lexicographically Maximum MEX Array bằng Python

- Link: https://leetcode.com/problems/lexicographically-maximum-mex-array/
- Độ khó: **Hard**
- Chủ đề: Tham lam (greedy), MEX, mảng đếm / mảng đánh dấu, so sánh từ điển

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumMEX(self, nums: List[int]) -> List[int]:
```

## 1. Giới thiệu & mục tiêu học

Ta cắt mảng `nums` thành các **đoạn liên tiếp** từ trái sang phải; mỗi đoạn đóng góp **MEX** của nó vào mảng kết quả `result`. Ta cần `result` **lớn nhất theo thứ tự từ điển**.

**MEX** (minimum excluded) của một mảng là số nguyên không âm **nhỏ nhất không có mặt** trong mảng. Ví dụ `mex([0, 1, 3]) = 2`.

Bài này là một ví dụ rất đẹp về tham lam kèm **chứng minh trao đổi** (exchange argument): chỉ có một lựa chọn tại mỗi bước là "an toàn", và nó có thể được tìm bằng hai con trỏ trong `O(n)`.

Sau notebook này bạn sẽ:

1. Hiểu MEX, tính chất **đơn điệu** (MEX của tiền tố không giảm khi tiền tố dài ra).
2. Nắm luật so sánh từ điển của đề (có xét **độ dài**: mảng dài hơn lớn hơn nếu tiền tố giống nhau).
3. Chứng minh hai quy tắc tham lam: (i) phần tử đầu tiên phải là MEX của **toàn bộ phần còn lại**; (ii) cắt ở tiền tố **ngắn nhất** đạt MEX đó.
4. Cài đặt `O(n)` bằng mảng MEX hậu tố + mảng đánh dấu có "tem thời gian".
5. Viết 3 lời giải (vét cạn quy hoạch động, tham lam tính lại MEX `O(n²)`, tham lam `O(n)`) và đối chiếu bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Lặp lại cho đến khi `nums` rỗng:

1. Chọn `k` với `1 ≤ k ≤ len(nums)`.
2. Tính MEX của `k` phần tử đầu, thêm vào cuối `result`.
3. Xóa `k` phần tử đầu khỏi `nums`.

Trả về `result` lớn nhất theo thứ tự từ điển. `a` lớn hơn `b` nếu ở vị trí đầu tiên hai mảng khác nhau `a` có số lớn hơn; nếu một mảng là **tiền tố** của mảng kia thì mảng **dài hơn lớn hơn**.

**Ví dụ 1:** `nums = [0,1,0]` → `[2,1]` (lấy `[0,1]` có MEX 2, còn `[0]` có MEX 1).

**Ví dụ 2:** `nums = [1,0,2]` → `[3]` (lấy cả mảng, MEX 3).

**Ví dụ 3:** `nums = [3,1]` → `[0,0]` (mỗi phần tử một đoạn, MEX đều là 0).

**Ràng buộc:** `1 ≤ len(nums) ≤ 10^5`, `0 ≤ nums[i] ≤ 10^5`.

In [1]:
import random, time
from functools import lru_cache

EXAMPLES = [
    ([0, 1, 0], [2, 1]),
    ([1, 0, 2], [3]),
    ([3, 1], [0, 0]),
]
for nums, exp in EXAMPLES:
    print("nums =", nums, "-> kỳ vọng", exp)

nums = [0, 1, 0] -> kỳ vọng [2, 1]
nums = [1, 0, 2] -> kỳ vọng [3]
nums = [3, 1] -> kỳ vọng [0, 0]


## 3. Kiến thức nền

### 3.1. MEX và cách tính nhanh bằng `set`

MEX của mảng `a` = số nguyên không âm nhỏ nhất **không** thuộc `a`. Cách tính đơn giản: bỏ `a` vào `set`, rồi thử `0, 1, 2, ...` đến khi gặp số không có.

In [2]:
def mex(a):
    s = set(a)
    m = 0
    while m in s:
        m += 1
    return m

print(mex([0, 1, 3]))        # 2  (thiếu 2)
print(mex([1, 2]))           # 0  (không có 0)
print(mex([0, 1, 2, 3]))     # 4
print(mex([]))               # 0

2
0
4
0


### 3.2. MEX của tiền tố không bao giờ giảm

Nếu thêm một phần tử vào mảng, tập số có mặt chỉ **lớn thêm**, nên số nhỏ nhất vắng mặt chỉ **tăng hoặc giữ nguyên**. Vì vậy: `mex(nums[:1]) ≤ mex(nums[:2]) ≤ mex(nums[:3]) ≤ ...`. Đây là chìa khóa của bài.

In [3]:
nums = [1, 0, 3, 0, 2, 4]
print([mex(nums[:k]) for k in range(1, len(nums) + 1)])   # không giảm

[0, 2, 2, 2, 4, 5]


### 3.3. So sánh từ điển của `list` trong Python

Python so sánh hai `list` đúng theo luật của đề: so từng vị trí; nếu một list là tiền tố của list kia thì **list dài hơn lớn hơn**. Nhờ đó ta dùng được `max(...)` và `>` để vét cạn.

In [4]:
print([2, 1] > [2])           # True : tiền tố bằng nhau, dài hơn lớn hơn
print([3] > [2, 5, 5])        # True : khác ở vị trí đầu, 3 > 2
print([2, 1] > [2, 0, 9])     # True
print(max([[2], [2, 1], [1, 9]]))

True
True
True
[2, 1]


### 3.4. Mảng đánh dấu có "tem thời gian" (timestamp)

Để biết "giá trị `x` đã xuất hiện trong **đoạn hiện tại** chưa" mà không phải xóa cả mảng đánh dấu sau mỗi đoạn, ta ghi **số thứ tự đoạn** (tem) vào `seen[x]`: `seen[x] == stamp` nghĩa là `x` đã xuất hiện trong đoạn đang xét. Đổi sang đoạn mới chỉ cần `stamp += 1`.

Ngoài ra MEX của mảng độ dài `L` không thể lớn hơn `L`, nên với `n` phần tử ta chỉ cần quan tâm giá trị `≤ n`; giá trị lớn hơn bỏ qua. Mảng đánh dấu cỡ `n + 2` là đủ.

In [5]:
n = 5
seen = [0] * (n + 2)
stamp = 1
for x in [0, 7, 1, 0]:          # đoạn 1
    if x <= n:
        seen[x] = stamp
print(seen, "-> 0 và 1 có trong đoạn 1")
stamp = 2                        # sang đoạn mới: không cần xóa seen
print("0 có trong đoạn 2?", seen[0] == stamp)

[1, 1, 0, 0, 0, 0, 0] -> 0 và 1 có trong đoạn 1
0 có trong đoạn 2? False


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `nums = [0, 1, 0]`. Có `2^(n−1) = 4` cách cắt:

| Cách cắt | MEX từng đoạn | `result` |
|---|---|---|
| `[0] [1] [0]` | 1, 0, 1 | `[1,0,1]` |
| `[0] [1,0]` | 1, 2 | `[1,2]` |
| `[0,1] [0]` | 2, 1 | `[2,1]` |
| `[0,1,0]` | 2 | `[2]` |

Lớn nhất: `[2,1]`. Hai quan sát:

- Phần tử đầu **càng lớn càng tốt**, mà MEX của tiền tố tăng theo độ dài. Giá trị lớn nhất có thể là MEX của **toàn bộ** mảng (`= 2`).
- Có hai cách để phần tử đầu bằng 2: `[0,1]` và `[0,1,0]`. Chọn `[0,1]` (ngắn hơn) cho kết quả `[2,1]` **dài hơn** `[2]`.

In [6]:
def all_cuts(nums):
    n = len(nums)
    for mask in range(1 << (n - 1)):
        parts, start = [], 0
        for i in range(n - 1):
            if mask >> i & 1:
                parts.append(nums[start:i + 1])
                start = i + 1
        parts.append(nums[start:])
        yield parts

for parts in all_cuts([0, 1, 0]):
    print(parts, "->", [mex(p) for p in parts])

[[0, 1, 0]] -> [2]
[[0], [1, 0]] -> [1, 2]
[[0, 1], [0]] -> [2, 1]
[[0], [1], [0]] -> [1, 0, 1]


## 5. Cách 1 — Brute Force: quy hoạch động thử mọi vị trí cắt

**Ý tưởng.** Gọi `best(s)` = `result` lớn nhất cho phần `nums[s:]`. Thử mọi độ dài đoạn đầu `k`: `[mex(nums[s:s+k])] + best(s+k)`, rồi lấy `max` theo thứ tự từ điển. Điều này đúng vì khi phần tử đầu cố định, so sánh từ điển chuyển thành so sánh phần đuôi.

In [7]:
def brute_dp(nums):
    n = len(nums)

    @lru_cache(maxsize=None)
    def best(s):
        if s == n:
            return ()
        cand = []
        for k in range(1, n - s + 1):
            cand.append((mex(nums[s:s + k]),) + best(s + k))
        return max(cand)

    return list(best(0))

for nums, exp in EXAMPLES:
    got = brute_dp(nums)
    print(nums, "->", got)
    assert got == exp

[0, 1, 0] -> [2, 1]
[1, 0, 2] -> [3]
[3, 1] -> [0, 0]


**Độ phức tạp.** `O(n)` trạng thái × `O(n)` lựa chọn × `O(n)` tính MEX và nối tuple = `O(n^3)` hay hơn. Dùng cho `n ≤ 10`.

**Nút thắt.** Thử mọi vị trí cắt là thừa: tính đơn điệu của MEX cho thấy chỉ có **một** vị trí cắt đáng để cân nhắc.

## 6. Bước đột phá

**Quy tắc 1 — phần tử đầu = MEX toàn bộ phần còn lại.** Vì MEX tiền tố không giảm theo độ dài, giá trị lớn nhất của phần tử đầu là `M = mex(nums[s:])` (lấy cả phần còn lại). Phần tử đầu quyết định so sánh từ điển trước tiên, nên ta **bắt buộc** chọn đoạn đầu có MEX bằng `M`.

**Quy tắc 2 — cắt ở tiền tố ngắn nhất có MEX bằng `M`.**

*Bổ đề.* Với mọi mảng `X` và mọi tiền tố thêm `P`: `result_max(P + X) ≥ result_max(X)`.

*Chứng minh.* Lấy cách cắt tối ưu của `X` với các đoạn `c₁, c₂, ...`. Cắt `P + X` thành `P + c₁, c₂, ...`. MEX của `P + c₁` không nhỏ hơn MEX của `c₁` (thêm phần tử không làm MEX giảm), các đoạn còn lại giữ nguyên. Vậy kết quả mới **lớn hơn hoặc bằng** theo từ điển (cùng độ dài, phần tử đầu không giảm). Do đó kết quả tối ưu của `P + X` cũng `≥`. ∎

Áp dụng: các tiền tố có MEX `= M` đều cho phần tử đầu `M`; tiền tố **ngắn hơn** để lại phần đuôi **dài hơn** (có thêm đoạn `P` ở đầu), nên theo bổ đề cho kết quả `≥`. Vậy chọn tiền tố **ngắn nhất** đạt MEX `= M`.

**Trường hợp `M = 0`** (phần còn lại không có số `0`): mọi tiền tố có MEX `0`, tiền tố ngắn nhất là `k = 1`. Quy tắc vẫn đúng.

**Thuật toán.** Lặp: tính `M = mex(phần còn lại)`; quét từ trái cộng dồn đến khi các số `0..M−1` đều xuất hiện (với `M = 0` cắt ngay một phần tử); thêm `M` vào kết quả; bỏ đoạn đó đi.

Cần hai thứ nhanh:
- `mex(nums[s:])` cho mọi `s` — tính trước **MEX hậu tố** từ phải sang trái bằng một con trỏ chỉ tăng.
- Quét đoạn ngắn nhất để đủ `0..M−1` — mảng đánh dấu có tem thời gian; mỗi phần tử chỉ bị quét **một lần** trong toàn bộ thuật toán.

In [8]:
# Kiểm tra bổ đề bằng thực nghiệm: result_max(P + X) >= result_max(X)
rng = random.Random(1)
for _ in range(300):
    X = [rng.randint(0, 3) for _ in range(rng.randint(1, 5))]
    P = [rng.randint(0, 3) for _ in range(rng.randint(1, 3))]
    assert brute_dp(P + X) >= brute_dp(X), (P, X)
print("bổ đề đúng trên 300 cặp (P, X) ngẫu nhiên")

bổ đề đúng trên 300 cặp (P, X) ngẫu nhiên


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam, tính lại MEX mỗi bước (`O(n²)`)

**Giải thích từng đoạn.**

1. `M = mex(nums[s:])` bằng `set` — `O(n)`.
2. Quét tiền tố: dùng `set` `have`, thêm từng phần tử, dừng khi `mex(have) == M` (hoặc đếm đủ `0..M−1`). Với `M = 0` dừng ngay sau một phần tử.
3. Ghi `M`, `s += k`.

Đúng, nhưng mỗi bước tốn `O(n)` ngay cả khi đoạn chỉ có độ dài 1 → `O(n²)` (ví dụ toàn số `5`: `n` đoạn × `O(n)` mỗi lần tính lại MEX của phần còn lại).

In [9]:
def greedy_recompute(nums):
    n = len(nums)
    res, s = [], 0
    while s < n:
        M = mex(nums[s:])                  # MEX của toàn bộ phần còn lại  -- O(n)
        if M == 0:
            res.append(0)
            s += 1
            continue
        have = set()
        k = 0
        while mex(have) < M:               # cộng dồn tới khi tiền tố có MEX = M
            have.add(nums[s + k])
            k += 1
        res.append(M)
        s += k
    return res

for nums, exp in EXAMPLES:
    got = greedy_recompute(nums)
    print(nums, "->", got)
    assert got == exp

[0, 1, 0] -> [2, 1]
[1, 0, 2] -> [3]
[3, 1] -> [0, 0]


**Độ phức tạp:** `O(n²)` (mỗi bước tính lại MEX phần còn lại; kể cả vòng `while mex(have) < M` còn tốn thêm `O(M)` mỗi lần kiểm tra). **Không gian:** `O(n)`.

### 7.2. Cách 3 — Tham lam `O(n)` bằng MEX hậu tố và tem thời gian

**Giải thích từng đoạn.**

1. **MEX hậu tố** `suf[i] = mex(nums[i:])`: duyệt `i` từ `n−1` về `0`; đánh dấu `nums[i]` đã có (nếu `≤ n`), rồi đẩy con trỏ `m` lên chừng nào `m` còn có mặt. `m` không bao giờ giảm nên tổng `O(n)`.
2. **Quét tiến:** tại vị trí bắt đầu `s`, `M = suf[s]`. Nếu `M = 0`: thêm `0`, `s += 1`. Ngược lại duyệt `j` từ `s`: nếu `nums[j] < M` và chưa có tem đoạn này thì đánh tem, tăng `got`; khi `got == M` cắt tại `j`, thêm `M`, `s = j + 1`, tăng `stamp`.
3. Mỗi `j` bị duyệt đúng một lần trong tổng thể → `O(n)`.

In [10]:
def greedy_linear(nums):
    n = len(nums)
    # --- MEX hậu tố ---
    present = [False] * (n + 2)
    suf = [0] * (n + 1)
    m = 0
    for i in range(n - 1, -1, -1):
        if nums[i] <= n:
            present[nums[i]] = True
        while present[m]:
            m += 1
        suf[i] = m
    # --- quét tiến ---
    seen = [0] * (n + 2)
    stamp = 1
    res = []
    s = 0
    while s < n:
        M = suf[s]
        if M == 0:
            res.append(0)
            s += 1
            continue
        got = 0
        j = s
        while True:
            x = nums[j]
            if x < M and seen[x] != stamp:
                seen[x] = stamp
                got += 1
                if got == M:
                    break
            j += 1
        res.append(M)
        s = j + 1
        stamp += 1
    return res

for nums, exp in EXAMPLES:
    got = greedy_linear(nums)
    print(nums, "->", got)
    assert got == exp

[0, 1, 0] -> [2, 1]
[1, 0, 2] -> [3]
[3, 1] -> [0, 0]


**Độ phức tạp:** `O(n)` thời gian (hai lượt quét, mỗi phần tử một lần), `O(n)` không gian. Đây là lời giải nộp.

**Vì sao vòng `while True` không chạy vượt mảng?** `M = suf[s]` là MEX của `nums[s:]`, nên mọi số `0..M−1` đều có mặt ở đâu đó trong `nums[s:]`; vòng lặp dừng khi gặp số cuối cùng trong chúng.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [1, 0, 2, 0, 1, 0, 5, 7]` (`n = 8`).

Tính tay MEX hậu tố (đi từ phải sang trái, thêm từng số vào tập):

| `i` | `nums[i]` | tập hậu tố có | `suf[i]` |
|---|---|---|---|
| 7 | 7 | {7} | 0 |
| 6 | 5 | {5,7} | 0 |
| 5 | 0 | {0,5,7} | 1 |
| 4 | 1 | {0,1,5,7} | 2 |
| 3 | 0 | {0,1,5,7} | 2 |
| 2 | 2 | {0,1,2,5,7} | 3 |
| 1 | 0 | {0,1,2,5,7} | 3 |
| 0 | 1 | {0,1,2,5,7} | 3 |

Các bước tham lam:

1. `s = 0`, `M = 3`: cần đủ `0,1,2` → đọc `1, 0, 2` → cắt `[1,0,2]`, ghi `3`.
2. `s = 3`, `M = suf[3] = 2`: cần `0,1` → đọc `0, 1` → cắt `[0,1]`, ghi `2`.
3. `s = 5`, `M = 1`: cần `0` → đọc `0` → cắt `[0]`, ghi `1`.
4. `s = 6`, `M = 0` → ghi `0`; `s = 7`, `M = 0` → ghi `0`.

Kết quả `[3, 2, 1, 0, 0]`. Hãy chạy hàm trace để kiểm tra từng bước.

In [11]:
def trace_linear(nums):
    n = len(nums)
    present = [False] * (n + 2)
    suf = [0] * (n + 1)
    m = 0
    for i in range(n - 1, -1, -1):
        if nums[i] <= n:
            present[nums[i]] = True
        while present[m]:
            m += 1
        suf[i] = m
    print("suf =", suf[:n])
    seen = [0] * (n + 2)
    stamp = 1
    res, s = [], 0
    while s < n:
        M = suf[s]
        print(f"\nbắt đầu s={s}, còn lại {nums[s:]}, M = mex(còn lại) = {M}")
        if M == 0:
            print("  M = 0 -> cắt 1 phần tử, MEX 0")
            res.append(0)
            s += 1
            continue
        got, j = 0, s
        while True:
            x = nums[j]
            new = x < M and seen[x] != stamp
            if new:
                seen[x] = stamp
                got += 1
            print(f"  j={j}: x={x} {'mới' if new else 'bỏ qua'}  got={got}/{M}")
            if got == M:
                break
            j += 1
        print(f"  cắt đoạn nums[{s}:{j + 1}] = {nums[s:j + 1]} -> MEX {M}")
        res.append(M)
        s = j + 1
        stamp += 1
    print("\nkết quả:", res)
    return res

assert trace_linear([1, 0, 2, 0, 1, 0, 5, 7]) == brute_dp([1, 0, 2, 0, 1, 0, 5, 7]) == [3, 2, 1, 0, 0]

suf = [3, 3, 3, 2, 2, 1, 0, 0]

bắt đầu s=0, còn lại [1, 0, 2, 0, 1, 0, 5, 7], M = mex(còn lại) = 3
  j=0: x=1 mới  got=1/3
  j=1: x=0 mới  got=2/3
  j=2: x=2 mới  got=3/3
  cắt đoạn nums[0:3] = [1, 0, 2] -> MEX 3

bắt đầu s=3, còn lại [0, 1, 0, 5, 7], M = mex(còn lại) = 2
  j=3: x=0 mới  got=1/2
  j=4: x=1 mới  got=2/2
  cắt đoạn nums[3:5] = [0, 1] -> MEX 2

bắt đầu s=5, còn lại [0, 5, 7], M = mex(còn lại) = 1
  j=5: x=0 mới  got=1/1
  cắt đoạn nums[5:6] = [0] -> MEX 1

bắt đầu s=6, còn lại [5, 7], M = mex(còn lại) = 0
  M = 0 -> cắt 1 phần tử, MEX 0

bắt đầu s=7, còn lại [7], M = mex(còn lại) = 0
  M = 0 -> cắt 1 phần tử, MEX 0

kết quả: [3, 2, 1, 0, 0]


In [12]:
# Một ví dụ cho thấy nhiều đoạn: có cả M = 0 lẫn M > 0
assert trace_linear([1, 0, 1, 0, 2, 0, 1, 5]) == brute_dp([1, 0, 1, 0, 2, 0, 1, 5])

suf = [3, 3, 3, 3, 3, 2, 0, 0]

bắt đầu s=0, còn lại [1, 0, 1, 0, 2, 0, 1, 5], M = mex(còn lại) = 3
  j=0: x=1 mới  got=1/3
  j=1: x=0 mới  got=2/3
  j=2: x=1 bỏ qua  got=2/3
  j=3: x=0 bỏ qua  got=2/3
  j=4: x=2 mới  got=3/3
  cắt đoạn nums[0:5] = [1, 0, 1, 0, 2] -> MEX 3

bắt đầu s=5, còn lại [0, 1, 5], M = mex(còn lại) = 2
  j=5: x=0 mới  got=1/2
  j=6: x=1 mới  got=2/2
  cắt đoạn nums[5:7] = [0, 1] -> MEX 2

bắt đầu s=7, còn lại [5], M = mex(còn lại) = 0
  M = 0 -> cắt 1 phần tử, MEX 0

kết quả: [3, 2, 0]


## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def maximumMEX(self, nums: List[int]) -> List[int]:
        n = len(nums)
        # suf[i] = mex(nums[i:])
        present = [False] * (n + 2)
        suf = [0] * (n + 1)
        m = 0
        for i in range(n - 1, -1, -1):
            if nums[i] <= n:
                present[nums[i]] = True
            while present[m]:
                m += 1
            suf[i] = m

        seen = [0] * (n + 2)
        stamp = 1
        res = []
        s = 0
        while s < n:
            M = suf[s]
            if M == 0:                       # không còn số 0: mỗi phần tử một đoạn
                res.append(0)
                s += 1
                continue
            got = 0
            j = s
            while True:
                x = nums[j]
                if x < M and seen[x] != stamp:
                    seen[x] = stamp
                    got += 1
                    if got == M:
                        break
                j += 1
            res.append(M)
            s = j + 1
            stamp += 1
        return res

sol = Solution()
for nums, exp in EXAMPLES:
    got = sol.maximumMEX(nums)
    print(nums, "->", got)
    assert got == exp

[0, 1, 0] -> [2, 1]
[1, 0, 2] -> [3]
[3, 1] -> [0, 0]


## 10. Vì sao thuật toán đúng

**Bất biến.** Tại mỗi bước, phần còn lại `R = nums[s:]` và `result` đã tạo là tiền tố của một kết quả tối ưu toàn cục.

1. **Phần tử đầu phải là `M = mex(R)`.** Với mọi `k`, `mex(R[:k]) ≤ mex(R) = M` (tiền tố là tập con của `R`), và `k = len(R)` đạt `M`. So sánh từ điển xét phần tử đầu trước nên mọi kết quả có phần tử đầu `< M` thua một kết quả có phần tử đầu `M`.
2. **Trong các tiền tố có MEX `= M`, tiền tố ngắn nhất `P` là tốt nhất.** Gọi `P` là tiền tố ngắn nhất, `Q ⊋ P` là tiền tố dài hơn cũng có MEX `M`. Phần đuôi sau `P` là `Q' + T` (với `Q' = Q \ P` đứng đầu), phần đuôi sau `Q` là `T`. Theo bổ đề ở mục 6: `result_max(Q' + T) ≥ result_max(T)`. Do đó chọn `P` không tệ hơn chọn `Q`; cũng đúng khi `T` rỗng (kết quả `[M]` ngắn hơn `[M, ...]` vì mảng dài hơn lớn hơn khi tiền tố bằng nhau — đây khớp với bổ đề: `result_max(Q') ≥ []`, tức không rỗng).
3. **Tìm `P` đúng.** `P` là tiền tố ngắn nhất chứa đủ `0..M−1` (khi `M ≥ 1`) — điều kiện cần và đủ để `mex(P) ≥ M`; vì `P ⊆ R` nên `mex(P) ≤ M`, suy ra `= M`. Với `M = 0`, mọi tiền tố có MEX `0`, ngắn nhất là `k = 1`.
4. **MEX hậu tố và tem thời gian** chỉ là cách hiện thực nhanh, không đổi kết quả. Giá trị `> n` không bao giờ ảnh hưởng MEX (MEX `≤ n`), nên bỏ qua đúng.

Test vét cạn ở mục 12 xác nhận trên hàng trăm mảng ngẫu nhiên.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`.

| Bước | Thời gian | Ghi chú |
|---|---|---|
| MEX hậu tố | `O(n)` | con trỏ `m` chỉ tăng, tối đa `n + 1` lần |
| Quét tiến cắt đoạn | `O(n)` | mỗi chỉ số `j` được thăm đúng một lần trong cả thuật toán (đoạn kế tiếp bắt đầu từ `j + 1`) |
| Mảng `present`, `suf`, `seen` | — | kích thước `O(n)` |

**Tổng:** `O(n)` thời gian, `O(n)` không gian (mảng kết quả cũng tối đa `n` phần tử). Đủ cho `n = 10^5`.

## 12. Bộ test

Ví dụ đề, edge case (một phần tử, toàn `0`, không có `0`, giá trị lớn hơn `n`), phản ví dụ cho cách làm sai, và đối chiếu ngẫu nhiên giữa brute force, Cách 2, Cách 3, `Solution`.

In [14]:
sol = Solution()
for nums, exp in EXAMPLES:
    assert sol.maximumMEX(nums) == exp

assert sol.maximumMEX([0]) == [1]
assert sol.maximumMEX([5]) == [0]
assert sol.maximumMEX([0, 0, 0]) == [1, 1, 1]              # mỗi đoạn một số 0, MEX 1
assert sol.maximumMEX([7, 8, 9]) == [0, 0, 0]
assert sol.maximumMEX([100000, 0, 1]) == [2]               # giá trị lớn hơn n bị bỏ qua đúng
assert sol.maximumMEX([2, 1, 0]) == [3]
assert sol.maximumMEX([1, 0, 1, 0]) == [2, 2]
print("ví dụ + edge case OK")

ví dụ + edge case OK


In [15]:
# Đối chiếu ngẫu nhiên: brute DP vs hai cách tham lam vs Solution
rng = random.Random(3948)
for _ in range(500):
    n = rng.randint(1, 9)
    hi = rng.choice([1, 2, 3, 5, 12])
    nums = [rng.randint(0, hi) for _ in range(n)]
    exp = brute_dp(nums)
    assert greedy_recompute(nums) == exp, nums
    assert greedy_linear(nums) == exp, nums
    assert sol.maximumMEX(nums) == exp, nums
print("500 test nhỏ khớp vét cạn")

for _ in range(300):
    n = rng.randint(1, 200)
    hi = rng.choice([2, 5, 20, n + 5])
    nums = [rng.randint(0, hi) for _ in range(n)]
    assert greedy_recompute(nums) == greedy_linear(nums) == sol.maximumMEX(nums), nums
print("300 test vừa: Cách 2 = Cách 3 = Solution")

500 test nhỏ khớp vét cạn
300 test vừa: Cách 2 = Cách 3 = Solution


In [16]:
# Thời gian: dữ liệu lớn
rng = random.Random(1)
n = 10**5
cases = {
    "ngẫu nhiên [0, 50]": [rng.randint(0, 50) for _ in range(n)],
    "toàn số 5": [5] * n,
    "hoán vị 0..n-1": rng.sample(range(n), n),
    "0,1,0,1,...": [i % 2 for i in range(n)],
    "ngẫu nhiên [0, 1e5]": [rng.randint(0, 10**5) for _ in range(n)],
}
for name, arr in cases.items():
    t0 = time.time()
    r = sol.maximumMEX(arr)
    print(f"{name:22s} -> {len(r)} đoạn, đầu {r[:3]}  ({time.time() - t0:.2f}s)")
t0 = time.time(); greedy_recompute([5] * 3000); print("Cách 2 với 3000 số 5:", f"{time.time() - t0:.2f}s")

ngẫu nhiên [0, 50]     -> 443 đoạn, đầu [51, 51, 51]  (0.01s)


toàn số 5              -> 100000 đoạn, đầu [0, 0, 0]  (0.01s)


hoán vị 0..n-1         -> 1 đoạn, đầu [100000]  (0.03s)


0,1,0,1,...            -> 50000 đoạn, đầu [2, 2, 2]  (0.02s)
ngẫu nhiên [0, 1e5]    -> 100000 đoạn, đầu [0, 0, 0]  (0.01s)
Cách 2 với 3000 số 5: 0.03s


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cắt ngay khi MEX vừa tăng (tham lam "ngắn nhất để có MEX > 0")

**Code sai:** cắt đoạn ngay khi MEX của tiền tố `> 0`.
**Vì sao sai:** phần tử đầu phải là **MEX lớn nhất có thể** (của cả phần còn lại), không phải MEX nhỏ nhất `> 0`.
**Code đúng:** cắt khi đạt `M = mex(phần còn lại)`.

In [17]:
def wrong_cut_early(nums):
    res, s, n = [], 0, len(nums)
    while s < n:
        have, k = set(), 0
        while True:
            have.add(nums[s + k]); k += 1
            if mex(have) > 0 or s + k == n:      # SAI: cắt ngay khi MEX > 0
                break
        res.append(mex(have)); s += k
    return res

print("sai :", wrong_cut_early([0, 1, 0]), " đúng:", Solution().maximumMEX([0, 1, 0]))

sai : [1, 2]  đúng: [2, 1]


### Lỗi 2 — Lấy cả phần còn lại thành một đoạn

**Code sai:** `result = [mex(nums)]`.
**Vì sao sai:** `[2] < [2, 1]` vì mảng dài hơn lớn hơn khi tiền tố bằng nhau. Phải cắt ngắn nhất để còn phần đuôi.
**Code đúng:** cắt tại tiền tố ngắn nhất đạt MEX.

In [18]:
def wrong_take_all(nums):
    return [mex(nums)]                            # SAI

print("sai :", wrong_take_all([0, 1, 0]), " đúng:", Solution().maximumMEX([0, 1, 0]))

sai : [2]  đúng: [2, 1]


### Lỗi 3 — Chọn tiền tố **dài nhất** có MEX `M`

**Code sai:** mở rộng thêm các phần tử còn lại mà MEX vẫn bằng `M`.
**Vì sao sai:** làm mất cơ hội tạo thêm đoạn sau (bổ đề: phần đuôi dài hơn không tệ hơn).
**Code đúng:** dừng ngay khi đủ `0..M−1`.

In [19]:
def wrong_longest_prefix(nums):
    n, res, s = len(nums), [], 0
    while s < n:
        M = mex(nums[s:])
        k = 1
        while k < n - s and mex(nums[s:s + k + 1]) == M:   # SAI: kéo dài tối đa mà MEX vẫn M
            k += 1
        # (chỉ đúng nếu mex(nums[s:s+k]) == M; ở đây giữ nguyên để minh họa)
        res.append(mex(nums[s:s + k])); s += k
    return res

print("sai :", wrong_longest_prefix([0, 1, 0, 1]), " đúng:", Solution().maximumMEX([0, 1, 0, 1]))

sai : [2]  đúng: [2, 2]


### Lỗi 4 — Mảng đánh dấu cỡ `n + 1` bị `IndexError` vì giá trị lớn

`nums[i]` có thể lớn tới `10^5` dù `n` nhỏ. Cần bỏ qua giá trị `> n` (MEX không thể vượt `n`) hoặc dùng `set`.

In [20]:
nums = [100000, 0, 1]
present = [False] * (len(nums) + 2)
try:
    for x in nums:
        present[x] = True                  # SAI: x = 100000 vượt chỉ số
except IndexError as e:
    print("IndexError:", e)
for x in nums:                             # ĐÚNG: bỏ qua giá trị > n
    if x <= len(nums):
        present[x] = True
print("present =", present)

IndexError: list assignment index out of range
present = [True, True, False, False, False]


### Lỗi 5 — Tính lại MEX phần còn lại bằng `set` mỗi bước (TLE)

Mỗi lần `set(nums[s:])` tốn `O(n)`. Với mảng nhiều đoạn ngắn tổng thành `O(n²)`. Hãy tính **MEX hậu tố** một lần (xem Cách 3).

## 14. Mẹo & nhận dạng pattern

- **"MEX của tiền tố/đoạn"** → nhớ tính đơn điệu: thêm phần tử không làm MEX giảm. Thường dẫn tới tham lam hoặc hai con trỏ.
- **Tối đa hóa theo từ điển** → làm phần tử đầu lớn nhất trước, rồi mới nghĩ đến phần sau; đừng quên **độ dài** cũng tham gia so sánh (dài hơn lớn hơn khi tiền tố bằng nhau).
- **Chứng minh tham lam bằng trao đổi:** "thêm phần tử vào đầu không làm kết quả tối ưu tệ đi" là một bổ đề dùng lại được nhiều lần.
- **Tính MEX bằng con trỏ chỉ tăng:** mỗi lần thêm số, đẩy `m` lên khi `present[m]`.
- **Tem thời gian** thay cho việc xóa mảng đánh dấu giữa các đoạn: tránh `O(n)` mỗi đoạn.
- Với `n` phần tử, chỉ cần quan tâm giá trị `≤ n` khi tính MEX.

## 15. Biến thể

1. **Cực tiểu từ điển:** muốn `result` nhỏ nhất → ngược lại, thường lấy `k = 1` cho phần tử đầu rồi nghĩ lại (cần chứng minh riêng).
2. **Chỉ được chọn `k` thuộc một tập cho trước** hoặc đoạn có độ dài tối đa `L`: tham lam không còn đúng, cần DP.
3. **Tối đa hóa tổng các MEX** thay vì từ điển: bài DP (MEX tiền tố không giảm, nhưng đoạn càng ngắn thì càng nhiều đoạn).
4. **Số đoạn cố định `K`:** thêm một chiều đếm đoạn.
5. **MEX theo cửa sổ trượt:** dùng đếm tần suất + segment tree/`set` các số vắng.

In [21]:
# Biến thể 3 (minh họa, vét cạn): tổng lớn nhất của các MEX khi cắt tùy ý
def max_sum_of_mex(nums):
    n = len(nums)
    @lru_cache(maxsize=None)
    def best(s):
        if s == n:
            return 0
        return max(mex(nums[s:s + k]) + best(s + k) for k in range(1, n - s + 1))
    return best(0)

print(max_sum_of_mex([0, 1, 0]), "(cắt [0] [1] [0] -> 1+0+1 = 2 ; [0,1] [0] -> 3)")
print("lex-max của bài gốc:", Solution().maximumMEX([0, 1, 0]))

3 (cắt [0] [1] [0] -> 1+0+1 = 2 ; [0,1] [0] -> 3)
lex-max của bài gốc: [2, 1]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP vét cạn | thử mọi `k`, lấy `max` từ điển | `O(n³)`+ | `O(n²)` | kiểm chứng, `n ≤ 10` |
| 2. Tham lam, tính lại MEX | chọn `M = mex(còn lại)`, cắt ngắn nhất | `O(n²)` | `O(n)` | hiểu thuật toán; `n` nhỏ |
| 3. Tham lam `O(n)` | MEX hậu tố + tem thời gian | `O(n)` | `O(n)` | **lời giải nộp** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `mex([2, 0, 1, 5])` bằng bao nhiêu?

**Bài 2 (tính tay).** Với `nums = [1, 0, 2, 0]`, `result` lớn nhất theo từ điển là gì? (Tính MEX toàn mảng, rồi tiền tố ngắn nhất.)

**Bài 3 (điền TODO).** Hoàn thiện `my_suffix_mex(nums)` trả về mảng `suf` với `suf[i] = mex(nums[i:])` trong `O(n)`.

**Bài 4 (tự viết lại).** Viết `my_maximum_mex(nums)` theo Cách 3, không nhìn code nộp.

**Bài 5 (biến thể).** Viết `lexicographically_smallest_mex(nums)` (vét cạn) trả về `result` **nhỏ nhất** theo từ điển, rồi so với `result` lớn nhất cho `[0, 1, 0]`.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_suffix_mex(nums):
    # TODO 1: present = [False] * (len(nums) + 2); m = 0; suf = [0] * len(nums)
    # TODO 2: i đi từ cuối về đầu: nếu nums[i] <= len(nums) thì present[nums[i]] = True
    # TODO 3: while present[m]: m += 1 ; suf[i] = m
    pass

# Bài 4 — TODO
def my_maximum_mex(nums):
    # TODO: suf, rồi vòng lặp s: M = suf[s]; M == 0 -> cắt 1; ngược lại quét tới khi đủ 0..M-1 (tem thời gian)
    pass

# Bài 5 — TODO
def lexicographically_smallest_mex(nums):
    # TODO: lru_cache + min(...) thay cho max(...) trong brute_dp
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tìm số nhỏ nhất không có trong mảng.
- Mức 2: có `0, 1, 2`; còn `3`?
- Mức 3: `3` không có → MEX `= 3`.

**Bài 2.**
- Mức 1: MEX của cả mảng `[1,0,2,0]`.
- Mức 2: đủ `0,1,2` → `M = 3`; tiền tố ngắn nhất chứa `0,1,2` là `[1,0,2]`.
- Mức 3: phần còn lại `[0]` có MEX `1` → `[3, 1]`.

**Bài 3.**
- Mức 1: đi từ cuối về đầu.
- Mức 2: đánh dấu `present`, đẩy con trỏ `m`.
- Mức 3: `m` không bao giờ giảm nên tổng `O(n)`.

**Bài 4.**
- Mức 1: tái dùng `my_suffix_mex`.
- Mức 2: tem thời gian `seen[x] == stamp`.
- Mức 3: sau khi cắt, `stamp += 1`, `s = j + 1`.

**Bài 5.**
- Mức 1: đổi `max` thành `min` trong brute force.
- Mức 2: so sánh `list` hoạt động y hệt, dài hơn vẫn lớn hơn.
- Mức 3: với `[0,1,0]` các kết quả là `[1,0,1]`, `[1,2]`, `[2,1]`, `[2]` → nhỏ nhất là `[1,0,1]`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert mex([2, 0, 1, 5]) == 3
# Bài 2
assert Solution().maximumMEX([1, 0, 2, 0]) == [3, 1] == brute_dp([1, 0, 2, 0])

# Bài 3
def my_suffix_mex_ans(nums):
    n = len(nums)
    present = [False] * (n + 2)
    suf = [0] * n
    m = 0
    for i in range(n - 1, -1, -1):
        if nums[i] <= n:
            present[nums[i]] = True
        while present[m]:
            m += 1
        suf[i] = m
    return suf

rng = random.Random(8)
for _ in range(100):
    arr = [rng.randint(0, 6) for _ in range(rng.randint(1, 10))]
    assert my_suffix_mex_ans(arr) == [mex(arr[i:]) for i in range(len(arr))]
print("Bài 1–3 OK")

Bài 1–3 OK


In [24]:
# Bài 4
def my_maximum_mex_ans(nums):
    n = len(nums)
    suf = my_suffix_mex_ans(nums)
    seen = [0] * (n + 2)
    stamp, s, res = 1, 0, []
    while s < n:
        M = suf[s]
        if M == 0:
            res.append(0); s += 1; continue
        got, j = 0, s
        while True:
            x = nums[j]
            if x < M and seen[x] != stamp:
                seen[x] = stamp; got += 1
                if got == M:
                    break
            j += 1
        res.append(M); s = j + 1; stamp += 1
    return res

rng = random.Random(9)
for _ in range(200):
    arr = [rng.randint(0, 4) for _ in range(rng.randint(1, 9))]
    assert my_maximum_mex_ans(arr) == brute_dp(arr)

# Bài 5
def lexicographically_smallest_mex_ans(nums):
    n = len(nums)
    @lru_cache(maxsize=None)
    def best(s):
        if s == n:
            return ()
        return min((mex(nums[s:s + k]),) + best(s + k) for k in range(1, n - s + 1))
    return list(best(0))

assert lexicographically_smallest_mex_ans([0, 1, 0]) == [1, 0, 1]
assert Solution().maximumMEX([0, 1, 0]) == [2, 1]
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
suf[i] = mex(nums[i:])            # từ phải sang, con trỏ m chỉ tăng, bỏ qua giá trị > n
s = 0
while s < n:
    M = suf[s]
    if M == 0: res.append(0); s += 1; continue
    got = 0; j = s
    while True:                    # tiền tố ngắn nhất chứa đủ 0..M-1
        x = nums[j]
        if x < M and seen[x] != stamp: seen[x] = stamp; got += 1
        if got == M: break
        j += 1
    res.append(M); s = j + 1; stamp += 1
```

**Checklist trước khi submit:**

- [ ] Phần tử đầu mỗi bước = MEX của **toàn bộ** phần còn lại (không phải MEX nhỏ nhất > 0).
- [ ] Cắt ở tiền tố **ngắn nhất** (mảng dài hơn lớn hơn khi tiền tố bằng nhau).
- [ ] Trường hợp `M = 0`: cắt đúng một phần tử.
- [ ] Mảng đánh dấu cỡ `n + 2`, bỏ qua giá trị `> n`.
- [ ] Tem thời gian tăng sau mỗi đoạn (không xóa mảng).
- [ ] Đã đối chiếu brute force DP với `n ≤ 9` và nhiều giá trị nhỏ (để có nhiều số trùng).
- [ ] Chạy thử `n = 10^5` các dạng: toàn số giống nhau, hoán vị, ngẫu nhiên.